# Day 4 v2 — Model 07: PhoBERT-base-v2 Fine-tune (top-K layers)

**Architecture:** `vinai/phobert-base-v2` (RoBERTa VN, 12L, 768d, 135M) — partial unfreeze top-K layers → mean_pooling → price head + aux category head.

**Preprocessing:** Underthesea word segmentation (required by PhoBERT) → cached pkl.

**Techniques:** LLRD + EMA(0.999) + HuberLoss(delta=1.0) + AMP + CosineWarmup + aux head.

**Sweep:** keep_top_layers=4 → evaluate → nếu MAE > 78k → re-run với keep_top_layers=8.

**Dataset:** `SeanSunny/items_tv_v9` (train=269K, val=3926, test=3872)

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
uv add underthesea
```

Restart kernel sau khi sync xong.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
from pricer_vi_2.phobert_model import PhoBERTRunner, PHOBERT_BASE

MODEL_NAME = PHOBERT_BASE  # vinai/phobert-base-v2
CACHE_DIR = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)

print(f"Model: {MODEL_NAME}")
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Underthesea Word Segmentation Cache

PhoBERT tokenizer yêu cầu text đã qua word segmentation (Underthesea).
Chạy **một lần duy nhất** (~2-3h trên CPU cho 269K samples), sau đó dùng cache.

Cache dùng chung cho NB07 (base) và NB08 (large).

In [ ]:
# Word segmentation: load cache if exists, else compute (~2-3h)
# Cache shared with NB08 (phobert-large uses same segmentation)
SEG_TRAIN = CACHE_DIR / "phobert_seg_train.pkl"
SEG_VAL   = CACHE_DIR / "phobert_seg_val.pkl"
SEG_TEST  = CACHE_DIR / "phobert_seg_test.pkl"

from pricer_vi_2.phobert_model import word_segment

# Pre-compute and cache all splits (longest step)
_ = word_segment([i.summary for i in train], cache_path=SEG_TRAIN)
_ = word_segment([i.summary for i in val],   cache_path=SEG_VAL)
_ = word_segment([i.summary for i in test],  cache_path=SEG_TEST)
print("Word segmentation cache ready.")

## 3. Setup — keep_top_layers=4 (4/12 layers = 33% unfrozen)

- Unfreeze top 4/12 transformer layers + price/category heads
- LLRD: head lr=2e-5, mỗi layer thấp hơn × decay=0.9
- Trainable params: ~30M (encoder top-4) + ~1M (heads)

In [ ]:
runner = PhoBERTRunner(train, val)

runner.setup(
    model_name=MODEL_NAME,
    keep_top_layers=4,
    batch_size=64,
    max_length=256,
    base_lr=2e-5,
    weight_decay=0.02,
    llrd_decay=0.9,
    dropout=0.2,
    train_seg_cache=SEG_TRAIN,
    val_seg_cache=SEG_VAL,
)

## 4. Train — top-4 layers

Expected: ~15-20 min/epoch on RTX 3090 Ti (269K × 256 tokens, batch=64).
Val MAE evaluated on full 3926 samples per epoch using EMA model.

In [ ]:
history_top4 = runner.train(
    epochs=10,
    patience=3,
    huber_delta=1.0,
    aux_alpha=0.1,
    ema_decay=0.999,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
)

In [ ]:
plot_training_history(history_top4, title="PhoBERT-base top-4 layers")

## 5. Evaluate top-4 on 200 test samples

In [ ]:
def phobert_base_top4_pricer(item):
    return runner.inference(item)

results_top4 = evaluate(phobert_base_top4_pricer, test)
print(f"[top-4] MAE: {results_top4['mae']:.1f}k VND | R2: {results_top4['r2']:.1f}%")
print()
print(">>> Nếu MAE > 78k: chạy Section 6 (top-8 sweep)")
print(">>> Nếu MAE <= 78k: bỏ qua Section 6, chạy thẳng Section 7 (Save)")

## 5b. Save top-4 weights + predictions

In [ ]:
Path("weights").mkdir(exist_ok=True)
runner.save("weights/phobert_base_top4.pth")
print("Saved weights/phobert_base_top4.pth")

Path("val_predictions").mkdir(exist_ok=True)

print("Running val predictions (3926)...")
val_preds_top4 = runner.val_predictions()
with open("val_predictions/phobert_base_val.json", "w") as f:
    json.dump(val_preds_top4, f)

print("Running test predictions (3872)...")
test_preds_top4 = runner.test_predictions(test, seg_cache=SEG_TEST)
with open("val_predictions/phobert_base_test.json", "w") as f:
    json.dump(test_preds_top4, f)

print(f"Val: {len(val_preds_top4)} | Test: {len(test_preds_top4)}")

## 6. [Optional Sweep] keep_top_layers=8 — Chỉ chạy nếu top-4 MAE > 78k

Unfreeze top 8/12 layers = 67% encoder. Nhiều hơn trainable params (~55M) nhưng cũng nhiều rủi ro overfit hơn.
Word seg cache được tái dụng, chỉ re-tokenize (~3-5 min) không chạy lại Underthesea.

In [ ]:
# Re-use same runner instance: setup() reinitializes model but reuses word-seg cache
runner.setup(
    model_name=MODEL_NAME,
    keep_top_layers=8,
    batch_size=64,
    max_length=256,
    base_lr=2e-5,
    weight_decay=0.02,
    llrd_decay=0.9,
    dropout=0.2,
    train_seg_cache=SEG_TRAIN,
    val_seg_cache=SEG_VAL,
)

In [ ]:
history_top8 = runner.train(
    epochs=10,
    patience=3,
    huber_delta=1.0,
    aux_alpha=0.1,
    ema_decay=0.999,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
)

In [ ]:
plot_training_history(history_top8, title="PhoBERT-base top-8 layers")

In [ ]:
def phobert_base_top8_pricer(item):
    return runner.inference(item)

results_top8 = evaluate(phobert_base_top8_pricer, test)
print(f"[top-8] MAE: {results_top8['mae']:.1f}k VND | R2: {results_top8['r2']:.1f}%")
print(f"[top-4] MAE: {results_top4['mae']:.1f}k VND")
print(f"Improvement: {results_top4['mae'] - results_top8['mae']:.1f}k VND")

In [ ]:
# Save top-8 weights + overwrite predictions with top-8 (better model)
runner.save("weights/phobert_base_top8.pth")
print("Saved weights/phobert_base_top8.pth")

print("Running val predictions (3926)...")
val_preds_top8 = runner.val_predictions()
with open("val_predictions/phobert_base_val.json", "w") as f:
    json.dump(val_preds_top8, f)

print("Running test predictions (3872)...")
test_preds_top8 = runner.test_predictions(test, seg_cache=SEG_TEST)
with open("val_predictions/phobert_base_test.json", "w") as f:
    json.dump(test_preds_top8, f)

print("Overwritten: val_predictions/phobert_base_{val,test}.json with top-8")

## 7. Sanity Check

In [ ]:
sample = test[0]
pred = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred:.1f}k VND")
print(f"Error:   {abs(pred - sample.price):.1f}k VND")

# Verify checkpoint keys
best_pth = "weights/phobert_base_top8.pth" if Path("weights/phobert_base_top8.pth").exists() else "weights/phobert_base_top4.pth"
ckpt = torch.load(best_pth, map_location="cpu", weights_only=False)
expected = {"ema_state_dict", "y_mean", "y_std", "model_name", "keep_top_layers", "cat_classes"}
assert expected == set(ckpt.keys()), f"Unexpected keys: {set(ckpt.keys())}"
print(f"\nCheckpoint ({best_pth}) keys OK")
print(f"model_name={ckpt['model_name']} | keep_top_layers={ckpt['keep_top_layers']}")
print(f"y_mean={ckpt['y_mean']:.4f} | y_std={ckpt['y_std']:.4f}")
print(f"Categories ({len(ckpt['cat_classes'])}): {ckpt['cat_classes']}")